In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv('/content/Netflix_Cleaned.csv')

df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
# Step 1: Dataset Overview

print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isna().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (8790, 10)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB

Missing Values:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Duplicate Rows:
0


In [3]:
# Step 1: Content Type Analysis

type_counts = df['type'].value_counts()
type_percentage = df['type'].value_counts(normalize=True) * 100

type_summary = pd.DataFrame({
    'Count': type_counts,
    'Percentage': type_percentage
})

type_summary

,Count,Percentage
type,,
Movie,6126,69.692833
TV Show,2664,30.307167


In [4]:
# Step 1: Rating Analysis

rating_counts = df['rating'].value_counts()

rating_percentage = df['rating'].value_counts(normalize=True) * 100

rating_summary = pd.DataFrame({
    'Count': rating_counts,
    'Percentage': rating_percentage
})

rating_summary

,Count,Percentage
rating,,
TV-MA,3205,36.461889
TV-14,2157,24.539249
TV-PG,861,9.795222
R,799,9.089875
PG-13,490,5.574516
TV-Y7,333,3.788396
TV-Y,306,3.481229
PG,287,3.265074
TV-G,220,2.502844


In [5]:
# Step 1: Genre Analysis

genre_data = df['listed_in'].dropna().str.split(', ').explode()

genre_counts = genre_data.value_counts()

genre_percentage = genre_counts / len(df) * 100

genre_summary = pd.DataFrame({
    'Count': genre_counts,
    'Percentage of Titles': genre_percentage
})

genre_summary.head(15)

,Count,Percentage of Titles
listed_in,,
International Movies,2752,31.308305
Dramas,2426,27.599545
Comedies,1674,19.044369
International TV Shows,1349,15.346985
Documentaries,869,9.886234
Action & Adventure,859,9.772469
TV Dramas,762,8.668942
Independent Movies,756,8.600683
Children & Family Movies,641,7.292378


In [6]:
# Step 1: Country Analysis

country_counts = df['country'].value_counts()

country_missing = df['country'].isna().sum()

country_not_given = (df['country'] == 'Not Given').sum()

top_countries = country_counts[country_counts.index != 'Not Given'].head(10)

print(f"Missing country values: {country_missing}")
print(f"'Not Given' values: {country_not_given}")
print(f"Number of unique country values: {df['country'].nunique()}")

print("\nTop 10 Countries:")
print(top_countries)

Missing country values: 0
'Not Given' values: 287
Number of unique country values: 86

Top 10 Countries:
country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Mexico             138
Name: count, dtype: int64


In [7]:
# Step 1: Release Year Analysis

print("Minimum Release Year:", df['release_year'].min())
print("Maximum Release Year:", df['release_year'].max())
print("Number of Unique Release Years:", df['release_year'].nunique())

year_counts = df['release_year'].value_counts().sort_values(ascending=False)

print("\nTop 10 Release Years by Number of Titles:")
print(year_counts.head(10))

Minimum Release Year: 1925
Maximum Release Year: 2021
Number of Unique Release Years: 74

Top 10 Release Years by Number of Titles:
release_year
2018    1146
2019    1030
2017    1030
2020     953
2016     901
2021     592
2015     555
2014     352
2013     286
2012     236
Name: count, dtype: int64


In [8]:
# Step 1: Date Added Analysis

df['date_added'] = pd.to_datetime(df['date_added'])

print("Earliest Date Added:", df['date_added'].min())
print("Latest Date Added:", df['date_added'].max())
print("Missing Date Added Values:", df['date_added'].isna().sum())

added_by_year = df['date_added'].dt.year.value_counts().sort_index()

print("\nTitles Added to Netflix by Year:")
print(added_by_year)

Earliest Date Added: 2008-01-01 00:00:00
Latest Date Added: 2021-09-25 00:00:00
Missing Date Added Values: 0

Titles Added to Netflix by Year:
date_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     426
2017    1185
2018    1648
2019    2016
2020    1879
2021    1498
Name: count, dtype: int64


In [9]:
# Step 1: Duration Analysis

movie_duration = (
    df.loc[df['type'] == 'Movie', 'duration']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

tv_duration = (
    df.loc[df['type'] == 'TV Show', 'duration']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

print("Movie Duration (minutes)")
print("------------------------")
print(f"Average: {movie_duration.mean():.2f} minutes")
print(f"Minimum: {movie_duration.min()} minutes")
print(f"Maximum: {movie_duration.max()} minutes")

print("\nTV Show Duration (seasons)")
print("---------------------------")
print(f"Average: {tv_duration.mean():.2f} seasons")
print(f"Minimum: {tv_duration.min()} season")
print(f"Maximum: {tv_duration.max()} seasons")

Movie Duration (minutes)
------------------------
Average: 99.58 minutes
Minimum: 3 minutes
Maximum: 312 minutes

TV Show Duration (seasons)
---------------------------
Average: 1.75 seasons
Minimum: 1 season
Maximum: 17 seasons


In [10]:
# Step 1: Director Analysis

director_not_given = (df['director'] == 'Not Given').sum()

director_counts = (
    df.loc[df['director'] != 'Not Given', 'director']
    .value_counts()
)

print(f"'Not Given' directors: {director_not_given}")
print(f"Unique directors (excluding 'Not Given'): {director_counts.shape[0]}")

print("\nTop 10 Directors by Number of Titles:")
print(director_counts.head(10))

'Not Given' directors: 2588
Unique directors (excluding 'Not Given'): 4527

Top 10 Directors by Number of Titles:
director
Rajiv Chilaka             20
Raúl Campos, Jan Suter    18
Alastair Fothergill       18
Suhas Kadav               16
Marcus Raboy              16
Jay Karas                 14
Cathy Garcia-Molina       13
Youssef Chahine           12
Martin Scorsese           12
Jay Chapman               12
Name: count, dtype: int64


In [11]:
# Step 1: Final Data Integrity Check

print("Total rows:", len(df))
print("Unique show IDs:", df['show_id'].nunique())
print("Unique titles:", df['title'].nunique())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Total rows: 8790
Unique show IDs: 8790
Unique titles: 8787
Missing values: 0
Duplicate rows: 0


In [12]:
# Step 2: Key Content Trends and Patterns

# Content production by release year
release_trend = (
    df['release_year']
    .value_counts()
    .sort_index()
)

# Content added to Netflix by year
added_trend = (
    df['date_added']
    .dt.year
    .value_counts()
    .sort_index()
)

print("Release Year Trend")
print("------------------")
print(release_trend.tail(10))

print("\nContent Added to Netflix Trend")
print("-------------------------------")
print(added_trend)

Release Year Trend
------------------
release_year
2012     236
2013     286
2014     352
2015     555
2016     901
2017    1030
2018    1146
2019    1030
2020     953
2021     592
Name: count, dtype: int64

Content Added to Netflix Trend
-------------------------------
date_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     426
2017    1185
2018    1648
2019    2016
2020    1879
2021    1498
Name: count, dtype: int64


In [13]:
# Step 2: Calculate yearly growth and decline

added_change = added_trend.diff()
added_growth_pct = added_trend.pct_change() * 100

trend_summary = pd.DataFrame({
    'Titles Added': added_trend,
    'Yearly Change': added_change,
    'Growth Rate (%)': added_growth_pct
})

trend_summary.round(2)

,Titles Added,Yearly Change,Growth Rate (%)
date_added,,,
2008,2,NaN,NaN
2009,2,0.0,0.00
2010,1,-1.0,-50.00
2011,13,12.0,1200.00
2012,3,-10.0,-76.92
2013,11,8.0,266.67
2014,24,13.0,118.18
2015,82,58.0,241.67
2016,426,344.0,419.51


In [14]:
# Step 2: Identify key content trends

# Use complete years only for yearly growth comparisons
complete_years = added_trend.loc[added_trend.index <= 2020]

complete_change = complete_years.diff()

peak_added_year = complete_years.idxmax()
peak_added_count = complete_years.max()

largest_increase_year = complete_change[complete_change > 0].idxmax()
largest_increase = complete_change[largest_increase_year]

largest_decrease_year = complete_change.idxmin()
largest_decrease = complete_change.min()

print(f"Peak year for content added: {peak_added_year} ({peak_added_count} titles)")
print(f"Largest increase: {largest_increase_year} (+{int(largest_increase)} titles)")
print(f"Largest decrease: {largest_decrease_year} ({int(largest_decrease)} titles)")

print("\nRecent yearly content added:")
print(complete_years.tail(6))

Peak year for content added: 2019 (2016 titles)
Largest increase: 2017 (+759 titles)
Largest decrease: 2020 (-137 titles)

Recent yearly content added:
date_added
2015      82
2016     426
2017    1185
2018    1648
2019    2016
2020    1879
Name: count, dtype: int64


In [15]:
# Step 2: Key Trends Summary

top_content_type = type_counts.index[0]
top_content_type_count = type_counts.iloc[0]
top_content_type_percentage = type_percentage.iloc[0]

top_country = top_countries.index[0]
top_country_count = top_countries.iloc[0]

top_rating = rating_counts.index[0]
top_rating_count = rating_counts.iloc[0]

top_genre = genre_counts.index[0]
top_genre_count = genre_counts.iloc[0]

peak_release_year = year_counts.index[0]
peak_release_count = year_counts.iloc[0]

print("KEY CONTENT TRENDS")
print("=" * 30)

print(f"Dominant content type: {top_content_type} "
      f"({top_content_type_count} titles, {top_content_type_percentage:.2f}%)")

print(f"Top country: {top_country} ({top_country_count} titles)")

print(f"Most common rating: {top_rating} ({top_rating_count} titles)")

print(f"Most common genre: {top_genre} ({top_genre_count} titles)")

print(f"Peak release year: {peak_release_year} "
      f"({peak_release_count} titles)")

print(f"Peak year for content added: {peak_added_year} "
      f"({peak_added_count} titles)")

KEY CONTENT TRENDS
Dominant content type: Movie (6126 titles, 69.69%)
Top country: United States (3240 titles)
Most common rating: TV-MA (3205 titles)
Most common genre: International Movies (2752 titles)
Peak release year: 2018 (1146 titles)
Peak year for content added: 2019 (2016 titles)


In [16]:
# Step 3: Dashboard 1 - Content Overview

import plotly.graph_objects as go
from plotly.subplots import make_subplots

total_titles = len(df)
movies_count = type_counts['Movie']
tv_shows_count = type_counts['TV Show']

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'}],
        [{'type': 'domain'}, {'type': 'bar'}]
    ],
    subplot_titles=(
        'Total Titles',
        'Movies',
        'Content Type Distribution',
        'Rating Distribution'
    )
)

# KPI: Total Titles
fig.add_trace(
    go.Indicator(
        mode='number',
        value=total_titles
    ),
    row=1,
    col=1
)

# KPI: Movies
fig.add_trace(
    go.Indicator(
        mode='number',
        value=movies_count
    ),
    row=1,
    col=2
)

# Content Type Pie Chart
fig.add_trace(
    go.Pie(
        labels=type_counts.index,
        values=type_counts.values,
        hole=0.4
    ),
    row=2,
    col=1
)

# Rating Bar Chart
fig.add_trace(
    go.Bar(
        x=rating_counts.index,
        y=rating_counts.values,
        name='Titles'
    ),
    row=2,
    col=2
)

fig.update_layout(
    title_text='Netflix Content Overview Dashboard',
    height=800,
    showlegend=True
)

fig.show()

In [17]:
# Verify content type counts

print(df['type'].value_counts())
print("\nMovie count:", (df['type'] == 'Movie').sum())
print("TV Show count:", (df['type'] == 'TV Show').sum())

type
Movie      6126
TV Show    2664
Name: count, dtype: int64

Movie count: 6126
TV Show count: 2664


In [18]:
# Step 3: Dashboard 1 - Content Overview

movies_count = (df['type'] == 'Movie').sum()
tv_shows_count = (df['type'] == 'TV Show').sum()
total_titles = len(df)

type_counts = df['type'].value_counts()
rating_counts = df['rating'].value_counts()

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'}],
        [{'type': 'domain'}, {'type': 'bar'}]
    ],
    subplot_titles=(
        'Total Titles',
        'Movies',
        'Content Type Distribution',
        'Rating Distribution'
    )
)

fig.add_trace(
    go.Indicator(
        mode='number',
        value=total_titles
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Indicator(
        mode='number',
        value=movies_count
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Pie(
        labels=type_counts.index,
        values=type_counts.values,
        hole=0.4
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Bar(
        x=rating_counts.index,
        y=rating_counts.values,
        name='Titles'
    ),
    row=2,
    col=2
)

fig.update_layout(
    title_text='Netflix Content Overview Dashboard',
    height=800,
    showlegend=True
)

fig.show()

In [19]:
# Final check before rebuilding Dashboard

movie_count_check = df['type'].eq('Movie').sum()
tv_show_count_check = df['type'].eq('TV Show').sum()
total_titles_check = len(df)

print("Total Titles:", total_titles_check)
print("Movies:", movie_count_check)
print("TV Shows:", tv_show_count_check)

Total Titles: 8790
Movies: 6126
TV Shows: 2664


In [20]:
# Step 3: Dashboard 1 - Content Overview

fig_dashboard1 = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'}],
        [{'type': 'domain'}, {'type': 'bar'}]
    ],
    subplot_titles=[
        'Total Titles',
        'Movies',
        'Content Type Distribution',
        'Rating Distribution'
    ]
)

# KPI: Total Titles
fig_dashboard1.add_trace(
    go.Indicator(
        mode='number',
        value=total_titles_check
    ),
    row=1,
    col=1
)

# KPI: Movies
fig_dashboard1.add_trace(
    go.Indicator(
        mode='number',
        value=movie_count_check
    ),
    row=1,
    col=2
)

# Content Type Distribution
fig_dashboard1.add_trace(
    go.Pie(
        labels=['Movie', 'TV Show'],
        values=[movie_count_check, tv_show_count_check],
        hole=0.45,
        textinfo='percent'
    ),
    row=2,
    col=1
)

# Rating Distribution
fig_dashboard1.add_trace(
    go.Bar(
        x=rating_counts.index,
        y=rating_counts.values,
        name='Titles'
    ),
    row=2,
    col=2
)

fig_dashboard1.update_layout(
    title='Netflix Content Overview Dashboard',
    height=800,
    showlegend=True
)

fig_dashboard1.show()

In [21]:
# Fix KPI number formatting

fig_dashboard1.update_traces(
    selector=dict(type='indicator'),
    number=dict(valueformat='.0f')
)

fig_dashboard1.show()

In [22]:
# Dashboard 2: Prepare Geographic and Genre Data

# Top 10 countries excluding "Not Given"
top_10_countries = (
    country_counts[country_counts.index != 'Not Given']
    .head(10)
    .reset_index()
)

top_10_countries.columns = ['Country', 'Titles']


# Top 10 genres
top_10_genres = (
    genre_counts
    .head(10)
    .reset_index()
)

top_10_genres.columns = ['Genre', 'Titles']

print("Top 10 Countries:")
print(top_10_countries)

print("\nTop 10 Genres:")
print(top_10_genres)

Top 10 Countries:
          Country  Titles
0   United States    3240
1           India    1057
2  United Kingdom     638
3        Pakistan     421
4          Canada     271
5           Japan     259
6     South Korea     214
7          France     213
8           Spain     182
9          Mexico     138

Top 10 Genres:
                      Genre  Titles
0      International Movies    2752
1                    Dramas    2426
2                  Comedies    1674
3    International TV Shows    1349
4             Documentaries     869
5        Action & Adventure     859
6                 TV Dramas     762
7        Independent Movies     756
8  Children & Family Movies     641
9           Romantic Movies     616


In [23]:
# Step 3: Dashboard 2 - Geographic & Genre Overview

fig_dashboard2 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        'Top 10 Countries by Number of Titles',
        'Top 10 Genres by Number of Titles'
    )
)

# Countries
fig_dashboard2.add_trace(
    go.Bar(
        x=top_10_countries['Titles'],
        y=top_10_countries['Country'],
        orientation='h',
        name='Countries',
        hovertemplate='%{y}: %{x} titles<extra></extra>'
    ),
    row=1,
    col=1
)

# Genres
fig_dashboard2.add_trace(
    go.Bar(
        x=top_10_genres['Titles'],
        y=top_10_genres['Genre'],
        orientation='h',
        name='Genres',
        hovertemplate='%{y}: %{x} titles<extra></extra>'
    ),
    row=1,
    col=2
)

fig_dashboard2.update_xaxes(
    title_text='Number of Titles',
    row=1,
    col=1
)

fig_dashboard2.update_xaxes(
    title_text='Number of Titles',
    row=1,
    col=2
)

fig_dashboard2.update_yaxes(
    categoryorder='total ascending',
    row=1,
    col=1
)

fig_dashboard2.update_yaxes(
    categoryorder='total ascending',
    row=1,
    col=2
)

fig_dashboard2.update_layout(
    title='Netflix Geographic & Genre Overview',
    height=700,
    width=1200,
    showlegend=False
)

fig_dashboard2.show()

In [24]:
# Dashboard 3: Content Trends Over Time

fig_dashboard3 = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        'Netflix Titles by Release Year',
        'Netflix Titles Added by Year'
    )
)

# Release Year Trend
fig_dashboard3.add_trace(
    go.Scatter(
        x=release_trend.index,
        y=release_trend.values,
        mode='lines+markers',
        name='Release Year',
        hovertemplate='Year: %{x}<br>Titles: %{y}<extra></extra>'
    ),
    row=1,
    col=1
)

# Date Added Trend
fig_dashboard3.add_trace(
    go.Scatter(
        x=added_trend.index,
        y=added_trend.values,
        mode='lines+markers',
        name='Date Added',
        hovertemplate='Year: %{x}<br>Titles Added: %{y}<extra></extra>'
    ),
    row=2,
    col=1
)

fig_dashboard3.update_xaxes(
    title_text='Release Year',
    row=1,
    col=1
)

fig_dashboard3.update_yaxes(
    title_text='Number of Titles',
    row=1,
    col=1
)

fig_dashboard3.update_xaxes(
    title_text='Year Added to Netflix',
    row=2,
    col=1
)

fig_dashboard3.update_yaxes(
    title_text='Number of Titles Added',
    row=2,
    col=1
)

fig_dashboard3.update_layout(
    title='Netflix Content Trends Over Time',
    height=900,
    showlegend=True,
    hovermode='x unified'
)

fig_dashboard3.show()

In [25]:
# Step 4: Generate actionable business insights

total_titles = len(df)

movie_share = movie_count_check / total_titles * 100
tv_share = tv_show_count_check / total_titles * 100

top_country_count = top_countries.iloc[0]
top_country_share = top_country_count / total_titles * 100

top_3_countries_count = top_countries.head(3).sum()
top_3_countries_share = top_3_countries_count / total_titles * 100

top_genre_count = genre_counts.iloc[0]
top_genre_share = top_genre_count / total_titles * 100

tv_ma_count = rating_counts['TV-MA']
tv_14_count = rating_counts['TV-14']
mature_rating_share = (tv_ma_count + tv_14_count) / total_titles * 100

content_added_peak_year = added_trend.idxmax()
content_added_peak_count = added_trend.max()

content_added_2016 = added_trend.loc[2016]
content_added_2019 = added_trend.loc[2019]
growth_2016_to_2019 = (content_added_2019 - content_added_2016) / content_added_2016 * 100

content_added_2020 = added_trend.loc[2020]
change_2019_to_2020 = (
    (content_added_2020 - content_added_2019)
    / content_added_2019
    * 100
)

print("BUSINESS METRICS")
print("=" * 35)

print(f"Movie share: {movie_share:.2f}%")
print(f"TV Show share: {tv_share:.2f}%")

print(
    f"Top country: {top_countries.index[0]} "
    f"({top_country_count} titles, {top_country_share:.2f}% of all titles)"
)

print(
    f"Top 3 countries share: "
    f"{top_3_countries_share:.2f}% of all titles"
)

print(
    f"Top genre: {genre_counts.index[0]} "
    f"({top_genre_count} titles, {top_genre_share:.2f}% of titles)"
)

print(
    f"TV-MA + TV-14 share: "
    f"{mature_rating_share:.2f}% of all titles"
)

print(
    f"Peak content-added year: "
    f"{content_added_peak_year} ({content_added_peak_count} titles)"
)

print(
    f"Content added growth from 2016 to 2019: "
    f"{growth_2016_to_2019:.2f}%"
)

print(
    f"Change in content added from 2019 to 2020: "
    f"{change_2019_to_2020:.2f}%"
)

print(f"\nAverage Movie duration: {movie_duration.mean():.2f} minutes")
print(f"Average TV Show duration: {tv_duration.mean():.2f} seasons")

BUSINESS METRICS
Movie share: 69.69%
TV Show share: 30.31%
Top country: United States (3240 titles, 36.86% of all titles)
Top 3 countries share: 56.14% of all titles
Top genre: International Movies (2752 titles, 31.31% of titles)
TV-MA + TV-14 share: 61.00% of all titles
Peak content-added year: 2019 (2016 titles)
Content added growth from 2016 to 2019: 373.24%
Change in content added from 2019 to 2020: -6.80%

Average Movie duration: 99.58 minutes
Average TV Show duration: 1.75 seasons


## Actionable Business Insights

### 1. Maintain a strong Movie portfolio
Movies represent 69.69% of the dataset, indicating that Movies are the dominant content format.
Netflix should continue maintaining a strong movie library while selectively expanding TV Show content where demand supports it.

### 2. Strengthen international content strategy
The United States accounts for 36.86% of all titles, while the top three countries together account for 56.14%.
This indicates a strong geographic concentration, suggesting an opportunity to continue expanding content from underrepresented markets.

### 3. Prioritize mature-audience content carefully
TV-MA and TV-14 together represent 61.00% of all titles.
Netflix may benefit from maintaining a strong mature-audience catalog while balancing it with family and younger-audience content.

### 4. Focus on high-performing genres
International Movies, Dramas, and Comedies are among the most frequently represented genres.
These genres can remain important areas for content acquisition and production decisions.

### 5. Monitor content-addition trends
Content additions increased substantially from 2016 to 2019, reaching a peak of 2,016 titles in 2019.
The decline in 2020 suggests the need to monitor changes in content acquisition and release strategies.

### 6. Consider content variety and portfolio balance
The dataset contains a wide range of ratings, genres, countries, and content types.
Maintaining this diversity can help Netflix serve different audience segments and geographic markets.

# Netflix Business Insights Report

## 1. Executive Summary

This report provides a business-oriented analysis of the Netflix dataset containing 8,790 titles. The analysis covers content types, ratings, genres, countries, release years, content addition trends, and duration.

The dataset is dominated by Movies, while TV-MA and TV-14 are the most common ratings. International Movies, Dramas, and Comedies are among the most frequently represented genres. The United States is the largest contributing country in the dataset, and content additions increased substantially through 2019 before declining in 2020.

---

## 2. Dataset Overview

- Total titles: 8,790
- Total columns: 10
- Unique show IDs: 8,790
- Unique titles: 8,787
- Missing values: 0
- Duplicate rows: 0
- Release years covered: 1925–2021
- Unique release years: 74

---

## 3. Content Portfolio Analysis

Movies represent 6,126 titles (69.69%) of the dataset, while TV Shows represent 2,664 titles (30.31%).

This shows that Movies are the dominant content format in the analyzed dataset.

---

## 4. Geographic Analysis

The United States is the largest contributing country with 3,240 titles, followed by India with 1,057 and the United Kingdom with 638.

The top three countries together account for 56.14% of all titles, showing a strong geographic concentration within the dataset.

---

## 5. Rating Analysis

TV-MA is the most common rating with 3,205 titles (36.46%), followed by TV-14 with 2,157 titles (24.54%).

Together, TV-MA and TV-14 account for 61.00% of all titles in the dataset.

When comparing content types, TV Shows have a higher share of TV-MA content than Movies (42.91% vs 33.66%).

TV Shows also have a higher share of TV-14 content than Movies (27.40% vs 23.29%).

---

## 6. Genre Analysis

International Movies are the most frequently represented genre with 2,752 titles, followed by Dramas with 2,426 titles and Comedies with 1,674 titles.

Other frequently represented categories include International TV Shows, Documentaries, Action & Adventure, and TV Dramas.

Note: A title can belong to multiple genres, so genre percentages are not expected to sum to 100%.

---

## 7. Content Trends Over Time

The number of titles by release year increased substantially during the 2010s.

The highest number of titles by release year was recorded in 2018 with 1,146 titles.

Content additions to Netflix also increased significantly over time, reaching a peak of 2,016 titles added in 2019.

From 2016 to 2019, the number of titles added increased by 373.24%.

Content additions decreased by 6.80% from 2019 to 2020.

The 2021 content-added figure should be interpreted carefully because the dataset ends on September 25, 2021 and therefore does not represent a complete calendar year.

---

## 8. Duration Analysis

Movies have an average duration of 99.58 minutes, with durations ranging from 3 to 312 minutes.

TV Shows have an average duration of 1.75 seasons, with durations ranging from 1 to 17 seasons.

Movie duration and TV Show duration were analyzed separately because they use different units.

---

## 9. Key Business Insights

1. **Maintain a balanced content portfolio**
   
   Movies form the majority of the dataset, but TV Shows represent a substantial share of the catalog. Portfolio planning should consider both formats.

2. **Expand geographic diversity**
   
   The dataset is concentrated among a small number of leading countries. Increasing representation from additional markets could support broader geographic diversity.

3. **Balance audience segments**
   
   TV-MA and TV-14 represent a large share of the catalog. Maintaining this mature-content presence while preserving family and younger-audience categories can support portfolio balance.

4. **Monitor high-volume genres**
   
   International Movies, Dramas, and Comedies are among the most frequently represented genres and are important components of the observed content portfolio.

5. **Monitor content acquisition trends**
   
   The sharp increase in titles added between 2016 and 2019 indicates a period of substantial catalog expansion. Changes after the 2019 peak should be monitored when evaluating future content-acquisition strategies.

6. **Use data diversity in portfolio planning**
   
   The dataset covers multiple countries, content types, ratings, and genres. This diversity provides opportunities to structure content strategies around different audience and geographic segments.

---

## 10. Analytical Limitations

- The dataset describes catalog content but does not contain viewer watch-time, engagement, revenue, or customer-level audience behavior.
- Genre counts can include the same title in multiple categories.
- The `country` field contains 287 `Not Given` values.
- The `director` field contains 2,588 `Not Given` values.
- The 2021 content-added data is incomplete because the latest recorded date is September 25, 2021.
- Therefore, the findings describe patterns in the available dataset and should not be interpreted as direct measures of audience demand or business performance.

---

## 11. Conclusion

The analysis shows a large and diverse Netflix content catalog with a strong concentration in Movies, mature ratings, international content, and a limited number of leading countries.

The dashboards and analysis provide a consolidated view of Netflix's content portfolio, geographic distribution, audience-rating structure, genre composition, and historical content trends.

These findings can support further analysis when additional business data such as viewing behavior, engagement, revenue, or audience demographics is available.